# 🤗 HuggingFace 한국어 감성 분석 실습

이 노트북은 **HuggingFace Transformers 라이브러리**를 처음 접하는 분들을 위한 실습 자료입니다.           
2차 인사이콘이나 이후 실습, 과제 제작할때 조금이나마 도움이 되셨으면 합니다!

### 📌 오늘 사용할 모델
- **모델명**: `daekeun-ml/koelectra-small-v3-nsmc`
- **모델 링크**: https://huggingface.co/daekeun-ml/koelectra-small-v3-nsmc
- **역할**: 한국어 문장을 입력하면 → **긍정(Positive) / 부정(Negative)** 을 판별
- **학습 데이터**: 네이버 영화 리뷰 데이터셋 (NSMC, 약 20만 건)

---
## Step 1. 📦 라이브러리 설치

HuggingFace를 사용하려면 `transformers` 라이브러리가 필요합니다.

| 라이브러리 | 역할 |
|---|---|
| `transformers` | HuggingFace 핵심 라이브러리. 모델과 토크나이저 제공 |
| `torch` | 딥러닝 연산을 처리하는 PyTorch 백엔드 |

In [ ]:
# 필요한 라이브러리 설치
# '!' 는 Jupyter에서 터미널 명령어를 실행하는 기호입니다
!pip install transformers torch -q

print("✅ 설치 완료!")

---
## Step 2. 📥 라이브러리 불러오기 (import)

`pipeline`은 HuggingFace에서 제공하는 **가장 간단한 사용법**입니다.

모델 로드, 토큰화, 추론까지 **단 몇 줄로** 처리할 수 있습니다.

```
[일반적인 딥러닝]          [HuggingFace pipeline]
1. 모델 정의              →  pipeline() 한 줄로 끝!
2. 가중치 로드
3. 토크나이저 설정
4. 전처리
5. 추론
6. 후처리
```

In [ ]:
# transformers 라이브러리에서 pipeline 함수를 불러옵니다
from transformers import pipeline

print("✅ pipeline 불러오기 완료!")

---
## Step 3. 🔧 모델 로드하기

이제 HuggingFace Hub에서 모델을 불러옵니다.

### 🤔잠깐! 그럼 아까 inference에 있던 코드를 다 옮겨와야 하나요?

모델카드에 있는 inference 코드는 이 모델을 직접 사용하는 방법을 보여줍니다.

**pipeline은 이 inference 과정을 자동으로 처리해주는 쉬운 방법**이고,
직접 모델을 사용하는 방식은 inference 과정을 하나씩 코드로 작성하는 방법입니다.

따라서 모델 내부 흐름을 자세히 이해하고 싶다면,
모델카드의 inference 코드를 참고해서 AutoTokenizer와 AutoModelForSequenceClassification을 직접 불러오면 됩니다.

### 우리는 일단 pipeline을 써서 간단하게 실습해 볼 거예요~

### pipeline()의 파라미터 설명

```python
pipeline(
    task,     # 어떤 작업을 할 것인지  (예: 감성분석, 번역, 요약 등)
    model     # 어떤 모델을 쓸 것인지  (HuggingFace Hub의 모델 ID)
)
```

| 파라미터 | 우리가 사용하는 값 | 의미 |
|---|---|---|
| `task` | `"text-classification"` | 텍스트를 특정 카테고리로 분류 |
| `model` | `"daekeun-ml/koelectra-small-v3-nsmc"` | 한국어 영화 리뷰 감성분석 모델 |

> ⏳ 처음 실행 시 모델 파일을 다운로드하므로 **1~2분** 정도 걸릴 수 있습니다.

In [ ]:
# HuggingFace Hub에서 모델을 다운로드하고 pipeline을 생성합니다
# 처음 실행 시 모델 파일이 자동으로 다운로드됩니다 (~54MB)
classifier = pipeline(
    task="text-classification",                    # 텍스트 분류 태스크
    model="daekeun-ml/koelectra-small-v3-nsmc"     # 사용할 모델 ID
)

print("✅ 모델 로드 완료! 이제 감성 분석을 할 수 있습니다.")

---
## Step 4. 🧪 예시 문장으로 테스트해보기

모델이 잘 동작하는지 예시 문장으로 먼저 확인해봅니다.

### 결과값 해석

```python
[{'label': 'Positive', 'score': 0.99}]
#   ↑ 분류 결과        ↑ 확신도 (0~1, 높을수록 확실)
```

| label | 의미 |
|---|---|
| 1 | `Positive` |
| 2 | `Negative` |

In [ ]:
# 예시 1: 긍정적인 문장 테스트
positive_text = "이 영화 정말 감동적이고 배우들 연기가 너무 훌륭해요!"
result = classifier(positive_text)

print(f"입력 문장: {positive_text}")
print(f"분석 결과: {result}")

In [ ]:
# 예시 2: 부정적인 문장 테스트
negative_text = "스토리도 엉망이고 시간 낭비였어요. 최악의 영화입니다."
result = classifier(negative_text)

print(f"입력 문장: {negative_text}")
print(f"분석 결과: {result}")

In [ ]:
# 예시 3: 여러 문장을 한번에 분석하기 (리스트로 전달)
sentences = [
    "배우들의 케미가 정말 좋았어요 ㅎㅎ",
    "돈 아까워요. 극장에서 나오고 싶었습니다.",
    "처음엔 별로였는데 후반부에서 반전이 있어서 좋았어요"
]

results = classifier(sentences)

print("=" * 55)
for text, result in zip(sentences, results):
    label = result['label']
    score = result['score']
    print(f" [{label}] (확신도: {score:.2%})")
    print(f"   → {text}")
    print("-" * 55)

---
## Step 5. ✍️ 나만의 문장 직접 입력해보기

이제 직접 문장을 입력하고 결과를 확인해보세요!

> 💡 이 모델은 **영화 리뷰 데이터**로 학습되었지만, 일반적인 한국어 감성 분석에도 꽤 잘 동작합니다.

In [ ]:
# ✏️ 아래 따옴표 안에 원하는 문장을 입력하세요!
my_sentence = "여기에 원하는 문장을 입력해보세요~!"  # ← 이 부분을 수정하세요

# 모델에 입력하여 결과 받기
result = classifier(my_sentence)

label = result[0]['label']
score = result[0]['score']

# 결과를 보기 좋게 출력
print("=" * 50)
print(f"📝 입력 문장: {my_sentence}")
print("-" * 50)
if label == "Positive":
    print(f"😊 결과: 긍정 (Positive)")
else:
    print(f"😞 결과: 부정 (Negative)")
print(f"🎯 확신도: {score:.2%}")
print("=" * 50)